# **Практическая работа №9. Развертывание автоматизированной системы преобразования координатных данных**

## **Цель работы**

Развернуть автоматизированную систему преобразования координатных данных, разработанную в практической работе №8, с использованием архитектуры клиент-сервер, где бэкенд размещается на платформе render.com, а фронтенд реализуется с помощью Streamlit.


Сервис должен принимать Excel-файл, генерировать промежуточный отчет в Markdown и конвертировать его в итоговый MS Word (.docx) через Pandoc (консольная утилита для Linux) или [PyPandoc (библиотека с аналогичным функионалом на Python)](https://pypi.org/project/pypandoc/).




Для зачета этой работы достаточно продемонстрировать корректное взаимодействие отдельно запущенных бэкенда и фронтенда в локальной среде (на своем ПК).


## **Задачи**



1. Разработать бэкенд-сервис на основе FastAPI для преобразования координатных данных
2. Создать фронтенд-приложение с использованием Streamlit для удобного взаимодействия с пользователем
3. Развернуть бэкенд-сервис на платформе render.com
4. Развернуть фронтенд-приложение на Streamlit Cloud
5. Настроить взаимодействие между фронтендом и бэкендом
6. Оформить документацию и исходный код в репозитории GitHub

## **Требования к выполнению работы**

### **Задание №1. Разработка бэкенда (FastAPI)**


1. Разработать приложение на основе FastAPI для обработки координатных данных:
   - Создать эндпоинт для загрузки Excel-файла
   - Реализовать функцию преобразования координат согласно алгоритму из практической работы №7
   - Добавить эндпоинт для возвращения результатов в формате markdown
   - Обеспечить обработку ошибок и валидацию входных данных

2. Код бэкенда должен быть структурирован следующим образом:
   - main.py - основной файл приложения FastAPI
   - requirements.txt - зависимости проекта

In [ ]:
# main.py
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse
from io import BytesIO
import pandas as pd
import numpy as np
from typing import List
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Инициализация FastAPI приложения
app = FastAPI(
    title="Excel to Markdown API",
    description="API для обработки Excel файлов и генерации отчетов в формате Markdown",
    version="1.0.0"
)

@app.get("/")
def read_root():
    """Корневой маршрут, возвращающий информацию о сервисе"""
    return {
        "message": "Excel to Markdown API работает",
        "endpoints": {
            "/process-excel/": "Загрузка и обработка Excel-файла с генерацией отчета в формате Markdown",
            "/proccess-exl2word": "Получение отчёта в формате Word"
        }
    }

@app.post("/process-excel/")
async def process_excel(file: UploadFile = File(...)):
    """
    Обрабатывает загруженный Excel-файл и возвращает отчет в формате Markdown
    """
    # Проверка формата файла
    if not file.filename.endswith(('.xlsx', '.xls')):
        raise HTTPException(
            status_code=400,
            detail="Поддерживаются только файлы Excel (.xlsx, .xls)"
        )

    try:
        # Чтение содержимого файла
        contents = await file.read()
        buffer = BytesIO(contents)

        # Загрузка Excel-файла в DataFrame
        df = pd.read_excel(buffer)

        # Генерация отчета в формате Markdown на основе данных
        report = generate_markdown_report(df)

        # Создание байтового объекта для хранения отчета
        output = BytesIO(report.encode())
        output.seek(0)

        # Возвращаем отчет как скачиваемый файл
        filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
        return StreamingResponse(
            output,
            media_type="text/markdown",
            headers={"Content-Disposition": f"attachment; filename={filename}"}
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Ошибка обработки файла: {str(e)}")

@app.post("/process-exl2word/")
async def process_exl2word(file: UploadFile = File(...)):
    #Will update later
    pass


#Преобразовать формат репорта в соответствии с результатами функции преобразования
def generate_markdown_report(df: pd.DataFrame) -> str:
    """
    Генерирует отчет в формате Markdown на основе предоставленного DataFrame
    """
    # Создаем заголовок отчета
    report = f"# Отчет по анализу данных\n\n"
    report += f"Дата создания: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"

    # Основная информация о данных
    report += f"## Общая информация\n\n"
    report += f"- **Количество строк**: {df.shape[0]}\n"
    report += f"- **Количество столбцов**: {df.shape[1]}\n"
    report += f"- **Столбцы**: {', '.join(df.columns)}\n\n"

    # Статистика по числовым столбцам
    report += f"## Статистический анализ\n\n"

    # Проверяем наличие числовых столбцов
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_columns:
        report += f"### Числовые данные\n\n"
        stats_df = df[numeric_columns].describe().transpose()
        # Форматируем статистику в виде таблицы Markdown
        stats_table = "| Столбец | Количество | Среднее | Ст. отклонение | Мин | 25% | 50% | 75% | Макс |\n"
        stats_table += "| --- | --- | --- | --- | --- | --- | --- | --- | --- |\n"

        for column, row in stats_df.iterrows():
            stats_table += f"| {column} | {row['count']:.0f} | {row['mean']:.2f} | {row['std']:.2f} | {row['min']:.2f} | {row['25%']:.2f} | {row['50%']:.2f} | {row['75%']:.2f} | {row['max']:.2f} |\n"

        report += stats_table + "\n\n"

    # Анализ категориальных данных
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if categorical_columns:
        report += f"### Категориальные данные\n\n"

        for column in categorical_columns:
            value_counts = df[column].value_counts().head(5)  # Топ-5 значений
            report += f"#### {column}\n\n"

            # Создаем таблицу с частотами значений
            report += "| Значение | Количество | Процент |\n"
            report += "| --- | --- | --- |\n"

            for value, count in value_counts.items():
                percentage = (count / len(df)) * 100
                report += f"| {value} | {count} | {percentage:.2f}% |\n"

            report += "\n"

    # Анализ пропущенных значений
    report += f"## Анализ пропущенных значений\n\n"
    missing_values = df.isnull().sum()
    if missing_values.sum() > 0:
        report += "| Столбец | Пропущенные значения | Процент пропущенных |\n"
        report += "| --- | --- | --- |\n"

        for column, missing in missing_values.items():
            if missing > 0:
                percentage = (missing / len(df)) * 100
                report += f"| {column} | {missing} | {percentage:.2f}% |\n"

        report += "\n"
    else:
        report += "Пропущенные значения отсутствуют.\n\n"

    # Заключение
    report += "## Выводы\n\n"
    report += "На основе анализа данных можно сделать следующие выводы:\n\n"
    report += "1. Данные содержат информацию о " + str(df.shape[0]) + " записях с " + str(df.shape[1]) + " характеристиками.\n"

    if numeric_columns:
        # Находим столбец с наибольшим средним значением
        max_mean_column = df[numeric_columns].mean().idxmax()
        max_mean_value = df[numeric_columns].mean().max()
        report += f"2. Столбец '{max_mean_column}' имеет наибольшее среднее значение ({max_mean_value:.2f}).\n"

    if missing_values.sum() > 0:
        most_missing = missing_values.idxmax()
        most_missing_count = missing_values.max()
        report += f"3. Столбец '{most_missing}' имеет наибольшее количество пропущенных значений ({most_missing_count}).\n"

    report += "\n"

    return report

### **Задание №2. Разработка фронтенда (Streamlit)**



1. Создать интерфейс пользователя с помощью Streamlit:
   - Добавить возможность загрузки Excel-файла
   - Реализовать отображение параметров преобразования и их настройку
   - Создать визуализацию результатов преобразования
   - Добавить возможность скачивания отчета в формате markdown

2. Код фронтенда должен быть структурирован следующим образом:
   - app.py - основной файл приложения Streamlit
   - utils/ - директория с вспомогательными функциями (при необходимости)

In [ ]:
# app.py
import streamlit as st
import pandas as pd
import requests
import io
from urllib.parse import urljoin

# Конфигурация приложения (не трогать)
st.set_page_config(
    page_title="Excel Analyzer",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded",
)

# URL бэкенда на render.com
# Замените на свой URL после развертывания на render.com
BACKEND_URL = "https://excel-to-markdown-api.onrender.com"

# Функции для взаимодействия с бэкендом
def check_api_status(): #(Не менять)
    """Проверяем доступность API"""
    try:
        response = requests.get(BACKEND_URL, timeout=10)
        return response.status_code == 200
    except requests.RequestException:
        return False

#Прим.Н.: Подобным образом добавить функции остальных эндпоинтов
def process_excel(file):
    """Отправляем Excel-файл на обработку в API и получаем Markdown-отчет"""
    url = urljoin(BACKEND_URL, "/process-excel/")
    files = {"file": file}
    try:
        response = requests.post(url, files=files)
        if response.status_code == 200:
            return response.content.decode('utf-8')
        else:
            st.error(f"Ошибка при обработке файла: {response.text}")
            return None
    except requests.RequestException as e:
        st.error(f"Ошибка соединения с API: {str(e)}")
        return None

# Интерфейс приложения
def main():
    st.title("📊 Анализатор Excel файлов")
    st.markdown("""
    Этот инструмент позволяет загрузить Excel-файл и получить аналитический отчет
    в формате Markdown. Просто загрузите файл и нажмите кнопку "Анализировать".
    """)

    # Проверка статуса API (не менять)
    if not check_api_status():
        st.error("⚠️ Не удалось подключиться к API. Пожалуйста, проверьте соединение или попробуйте позже.")
        return

    # Загрузка файла (не менять)
    uploaded_file = st.file_uploader("Выберите Excel файл", type=['xlsx', 'xls'])

    if uploaded_file is not None:
        # Показываем предварительный просмотр данных
        try:
            df = pd.read_excel(uploaded_file)
            st.subheader("Предварительный просмотр данных")
            st.dataframe(df.head(5))

            # Получение основных статистик для информации
            st.subheader("Базовая информация")
            col1, col2, col3 = st.columns(3)
            col1.metric("Строки", df.shape[0])
            col2.metric("Столбцы", df.shape[1])
            col3.metric("Пропущенные значения", df.isna().sum().sum())

            # Сбрасываем указатель файла для повторного чтения
            uploaded_file.seek(0)


            #ПОДОГНАТЬ под текущее задание
            if st.button("Анализировать"):
                with st.spinner("Обрабатываем данные..."):
                    markdown_report = process_excel(uploaded_file)

                if markdown_report:
                    st.success("Отчет успешно создан!")

                    # Показываем отчет в интерфейсе
                    st.subheader("Отчет")
                    st.markdown(markdown_report)

                    # Предоставляем возможность скачать отчет
                    st.download_button(
                        label="Скачать отчет",
                        data=markdown_report,
                        file_name="report.md",
                        mime="text/markdown",
                    )

        except Exception as e:
            st.error(f"Ошибка при чтении файла: {str(e)}")

if __name__ == "__main__":
    main()

### **Задание №3. Развертывание системы**



1. Развернуть бэкенд на платформе render.com:
   - Создать новый Web Service на render.com
   - Настроить переменные окружения (если необходимо)
   - Проверить работоспособность развернутого сервиса

2. Развернуть фронтенд на Streamlit Cloud:
   - Создать учетную запись на Streamlit Cloud (если отсутствует)
   - Настроить репозиторий для развертывания
   - Проверить работоспособность развернутого приложения

### **Задание №4. Тестирование и документация**


1. Протестировать работу системы на различных наборах данных:
   - Создать тестовый набор данных с использованием скрипта генерации
   - Проверить корректность преобразования координат
   - Убедиться в правильности формирования отчета

2. Оформить документацию:
   - Создать README.md с описанием проекта и инструкциями по установке
   - Добавить комментарии к коду
   - Подготовить пример использования системы

   Пример оформления:
  1. https://github.com/Alexandre77777/dpo_classification
  2. https://github.com/Alexandre77777/dpo_regression